In [1]:
import sys
import os

# 1. Dynamically inject the local venv packages into Python path
# This bypasses the VS Code kernel selection bug
LOCAL_VENV_PATH = os.path.join("..", "venv", "Lib", "site-packages")
if os.path.exists(LOCAL_VENV_PATH):
    sys.path.insert(0, os.path.abspath(LOCAL_VENV_PATH))
    print("[SUCCESS] Local venv site-packages path injected into memory!")

# 2. Try to import pandas to verify if it works now
try:
    import pandas as pd
    print(f"[SUCCESS] Pandas library loaded correctly. Version: {pd.__version__}")
    
    # 3. Test loading our project data
    DATA_PATH = os.path.join("..", "data", "raw", "GermanCredit.csv")
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
        print(f"[SUCCESS] Dataset loaded! Dimensions: {df.shape}")
    else:
        print(f"[ERROR] Could not find 'GermanCredit.csv' at: {DATA_PATH}")
except ImportError:
    print("[ERROR] Pandas is not installed in the current environment path.")


[SUCCESS] Local venv site-packages path injected into memory!
[SUCCESS] Pandas library loaded correctly. Version: 2.2.2
[SUCCESS] Dataset loaded! Dimensions: (1000, 21)


# Phase 1: Data Loading and Structural Inspection

In this section, we will load the German Credit Risk dataset and analyze its structural integrity, identifying data types and the distribution of our target variable (`credit_risk`).


In [2]:
print("---- DataFrame Structural Summary ----")
df.info()

---- DataFrame Structural Summary ----


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   status                   1000 non-null   object
 1   duration                 1000 non-null   int64 
 2   credit_history           1000 non-null   object
 3   purpose                  1000 non-null   object
 4   amount                   1000 non-null   int64 
 5   savings                  1000 non-null   object
 6   employment_duration      1000 non-null   object
 7   installment_rate         1000 non-null   int64 
 8   personal_status_sex      1000 non-null   object
 9   other_debtors            1000 non-null   object
 10  present_residence        1000 non-null   int64 
 11  property                 1000 non-null   object
 12  age                      1000 non-null   int64 
 13  other_installment_plans  1000 non-null   object
 14  housing                  1000 non-null   

In [3]:
# Calculate the absolute and percentage distributions fo the target variable
target_distribution = df['credit_risk'].value_counts()

In [4]:
target_percentage = df['credit_risk'].value_counts(normalize=True) * 100

In [5]:
print(target_distribution)
print(target_percentage)

credit_risk
1    700
0    300
Name: count, dtype: int64
credit_risk
1    70.0
0    30.0
Name: proportion, dtype: float64


# Phase 2: Exploratory Data Analysis (EDA) - Categorical Features

In this section, we isolate and analyze all categorical columns (`object` type) to understand their unique values and distributions before applying encoding techniques.


In [7]:
# Select and display only the categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

In [8]:
print(f"Total categorical features found: {len(categorical_cols)}")
print(f'\nList of categorical features: ')
print(list(categorical_cols))

Total categorical features found: 13

List of categorical features: 
['status', 'credit_history', 'purpose', 'savings', 'employment_duration', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']


In [9]:
# 1. Identify all categorical columns (object type)
categorical_cols = df.select_dtypes(include=["object"]).columns

# 2. Print the unique values for the first 3 categorical features to inspect them
print("--- Inspecting Sample Categorical Features ---")
for col in list(categorical_cols)[:3]:
    print(f"\nUnique values for column '{col}':")
    print(df[col].unique())


--- Inspecting Sample Categorical Features ---

Unique values for column 'status':
['... < 100 DM' '0 <= ... < 200 DM' 'no checking account'
 '... >= 200 DM / salary for at least 1 year']

Unique values for column 'credit_history':
['critical account/other credits existing'
 'existing credits paid back duly till now'
 'delay in paying off in the past'
 'no credits taken/all credits paid back duly'
 'all credits at this bank paid back duly']

Unique values for column 'purpose':
['domestic appliances' 'retraining' 'radio/television' 'car (new)'
 'car (used)' 'others' 'repairs' 'education' 'furniture/equipment'
 'business']


## 2.1.1 Analyzing Credit History vs Credit Risk

I will now create a cross-tabulation between `credit_history` and our target variable `credit_risk` to analyze how past payment behavior impacts the final classification.


In [12]:
# Cerate a corsso-tabulation between credit history and credit risk
history_vs_risk = pd.crosstab(df["credit_history"], df["credit_risk"], normalize="index") * 100

In [13]:
# Display the resuylt sorted by bad payers percentage(class 0)
print('---- Credit History vs Credit Risk (%) ----')
print(history_vs_risk.round(2))

---- Credit History vs Credit Risk (%) ----
credit_risk                                      0      1
credit_history                                           
all credits at this bank paid back duly      57.14  42.86
critical account/other credits existing      17.06  82.94
delay in paying off in the past              31.82  68.18
existing credits paid back duly till now     31.89  68.11
no credits taken/all credits paid back duly  62.50  37.50


## 2.1.2 Analyzing Employment Duration vs Credit Risk

I analyze how the length of time a client has been employed impacts their credit risk profile using a normalized cross-tabulation.


In [14]:
# Cross-tabulate employment duration with the credit risk target
employment_vs_risk = pd.crosstab(df['employment_duration'], df['credit_risk'], normalize='index') * 100

In [15]:
print('---- Employment Duration vs Credit Risk (%) ----')
print(employment_vs_risk.round(2))

---- Employment Duration vs Credit Risk (%) ----
credit_risk              0      1
employment_duration              
... < 1 year         40.70  59.30
... >= 7 years       25.30  74.70
1 <= ... < 4 years   30.68  69.32
4 <= ... < 7 years   22.41  77.59
unemployed           37.10  62.90


### 2.1.3 Housing Status vs Credit Risk Analysis

In this step, I will cross-tabulate the `housing` variable against the `credit_risk` target. The objective is to understand how property ownership (rented, owned, or free housing) correlates with credit default rates.


In [16]:
# Analyze the relationship between housing types and credit risk
housing_vs_risk = pd.crosstab(df["housing"], df["credit_risk"], normalize="index") * 100

print("--- Housing Status vs Credit Risk (%) ---")
print(housing_vs_risk.round(2))


--- Housing Status vs Credit Risk (%) ---
credit_risk      0      1
housing                  
for free     40.74  59.26
own          26.09  73.91
rent         39.11  60.89
